## DECORATORS

### Decorators allows modifying a functions behaviour ***without changing its source code***, using higher order function.

### **Common uses:** *logging, authentication, caching, timing*

Why decorator?
- No repeated auth code
- Security logic is enforced automatically

**Flow: func() → wrapper() → original func()**

### Authentication / Authorization (Web APIs)

In [ ]:
def authenticate(func):
    def wrapper(user, *args, **kwargs):
        if not user.get("is_authenticated"): # get is used here to avoid API crash
            '''
            If we use -> if not user["is_authenticated"] THEN we might get KeyError and the API would have crashed.
            '''
            raise PermissionError("Unauthorized")
        return func(user, *args, **kwargs)
    return wrapper

@authenticate
def get_user_profile(user):
    return f"Profile of {user['name']}"

user = {"name": "Aviral", "is_authenticated": True}
print(get_user_profile(user))

Profile of Aviral


### Retry logic (very common in data engineering)

```mermaid
flowchart TD
    A(Call fetch_api) --> B(wrapper starts)
    B --> C(attempt = 1)
    C --> D{Call original function}
    D -->|Success| E(Return result)
    D -->|Exception| F(Print attempt failed)
    F --> G(Sleep for delay seconds)
    G --> H{More retries left?}
    H -->|Yes| I(attempt += 1)
    I --> D
    H -->|No| J(Raise All retries failed)


In [ ]:
import time

def retry(retries=3, delay=1):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for attempt in range(retries):
                try:
                    return func(*args, **kwargs) # Decorator function i.e fetch_api()
                except Exception as e:
                    print(f"Attempt {attempt + 1} failed")
                    time.sleep(delay) # Gives time to breathe.....
            raise Exception("All retries failed")
        return wrapper
    return decorator

@retry(retries=3, delay=2)
def fetch_api():
    print("Calling API")
    raise Exception("API down")

fetch_api()

Calling API
Attempt 1 failed
Calling API
Attempt 2 failed
Calling API
Attempt 3 failed


Exception: All retries failed

Why decorator?
- Retry logic is reusable
- No need to write try/except everywhere

### Caching (Performance boost)

In [ ]:
def cache(func):
    memory = {}                # stores computed results

    def wrapper(*args):
        if args in memory:     # check if result already exists
            return memory[args]

        result = func(*args)   # compute result
        memory[args] = result  # save in cache
        return result

    return wrapper

@cache
def slow_square(n):
    print("Computing...")
    return n * n

print(slow_square(3))
print(slow_square(4))
print(slow_square(4))  # cached


Computing...
16
16
Computing...
9


```mermaid
flowchart TD
    A[Call slow_square] --> B[wrapper starts]
    B --> C{args in cache?}
    C -->|Yes| D[Return cached result]
    C -->|No| E[Call original function]
    E --> F[Compute result]
    F --> G[Store result in cache]
    G --> H[Return result]

What return does
- Sends value back to caller
- Stops the function completely
- Nothing below it runs

**So continue is not needed.**